In [3]:
import sys
!{sys.executable} -m pip install qiskit

from qiskit import QuantumCircuit
import numpy as np

# Step 1: Set up the Quantum Circuit
# We need 2 qubits for the pixel positions (to hold 4 pixels: 00, 01, 10, 11)
# We need 1 qubit for the color (the theta angle)
# Total = 3 qubits
qc = QuantumCircuit(3)

# Step 2: Create the Spatial Superposition
# We apply Hadamard (H) gates to the position qubits (qubits 0 and 1).
# This spreads our circuit across all 4 pixel locations simultaneously!
qc.h(0)
qc.h(1)

# Step 3: Define our 2x2 Classical Image Colors
# We represent colors as angles (theta).
# Let's create a simple image: Dark, Light, Light, Dark
theta_00 = np.pi / 2  # Pixel 00 (Top-left)
theta_01 = 0.0        # Pixel 01 (Top-right)
theta_10 = 0.0        # Pixel 10 (Bottom-left)
theta_11 = np.pi / 2  # Pixel 11 (Bottom-right)

# Step 4: Encode the Colors into the Quantum State (FRQI Protocol)
# We use Multi-Controlled Y-Rotations (mcry) to change the color qubit (qubit 2)
# ONLY when the position qubits match the specific pixel we want to color.

# Pixel 00 Encoding
qc.x(0) # Flip 0 to 1 so the control activates
qc.x(1) # Flip 0 to 1 so the control activates
qc.mcry(2 * theta_00, [0, 1], 2) # Apply color rotation
qc.x(1) # Undo flip
qc.x(0) # Undo flip

# Pixel 01 Encoding
qc.x(1)
qc.mcry(2 * theta_01, [0, 1], 2)
qc.x(1)

# Pixel 10 Encoding
qc.x(0)
qc.mcry(2 * theta_10, [0, 1], 2)
qc.x(0)

# Pixel 11 Encoding (No X gates needed because it's already 11)
qc.mcry(2 * theta_11, [0, 1], 2)

# Step 5: Show the Circuit
print("Phase 1: FRQI Image Encoding Circuit built successfully!")
print("Here is your circuit diagram:")
print(qc.draw('text'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.3 MB/s eta 0:00:00
Phase 1: FRQI Image Encoding Circuit built successfully!
Here is your circuit diagram:
        ┌───┐   ┌───┐                        ┌───┐                          »
q_0: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├─────────■─────────────■──»
        ├───┤   ├───┤  │                │    ├───┤  ┌───┐  │             │  »
q_1: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├──┤ X ├──■─────────────■──»
     ┌──┴───┴──┐└───┘┌─┴─┐┌──────────┐┌─┴─┐┌─┴───┴─┐└───┘┌─┴─┐┌───────┐┌─┴─┐»
q_2: ┤ Ry(π/2) ├─────┤ X ├┤ Ry(-π/2) ├┤ X ├┤ Ry(0) ├─────┤ X ├┤ Ry(0) ├┤ X ├»
     └─────────┘     └───┘└──────────┘└───┘└───────┘     └───┘└───────┘└───┘»
«       ┌───┐                        ┌───┐                         
«q_0: ──┤ X ├────■─────────────■─────┤ X ├─────■────────────────■─

In [4]:
from qiskit.circuit.library import QFT

# --- PHASE 2: THE SORTING MACHINE ---

# Step 1: Create a visual barrier so our circuit diagram is easy to read
qc.barrier()

# Step 2: Build the Quantum Fourier Transform (QFT)
# We only need a 2-qubit QFT because we are only sorting the spatial pixels (qubits 0 and 1).
# The color qubit (qubit 2) just goes along for the ride.
qft_circuit = QFT(num_qubits=2, do_swaps=True, inverse=False, name='QFT')

# Step 3: Attach the QFT to our main circuit
qc.append(qft_circuit, [0, 1])

# Step 4: Decompose the QFT
# (This unboxes the QFT so we can see the actual H gates and controlled-phase gates inside it)
qc = qc.decompose('QFT')

# Step 5: Show the updated circuit
print("\n--- Phase 2 Complete ---")
print("QFT Sorting Machine added successfully!")
print("Here is your updated circuit diagram:")
print(qc.draw('text'))

/tmp/ipykernel_439/1261224046.py:11: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qft_circuit = QFT(num_qubits=2, do_swaps=True, inverse=False, name='QFT')



--- Phase 2 Complete ---
QFT Sorting Machine added successfully!
Here is your updated circuit diagram:
        ┌───┐   ┌───┐                        ┌───┐                          »
q_0: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├─────────■─────────────■──»
        ├───┤   ├───┤  │                │    ├───┤  ┌───┐  │             │  »
q_1: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├──┤ X ├──■─────────────■──»
     ┌──┴───┴──┐└───┘┌─┴─┐┌──────────┐┌─┴─┐┌─┴───┴─┐└───┘┌─┴─┐┌───────┐┌─┴─┐»
q_2: ┤ Ry(π/2) ├─────┤ X ├┤ Ry(-π/2) ├┤ X ├┤ Ry(0) ├─────┤ X ├┤ Ry(0) ├┤ X ├»
     └─────────┘     └───┘└──────────┘└───┘└───────┘     └───┘└───────┘└───┘»
«       ┌───┐                        ┌───┐                          ░ ┌──────┐
«q_0: ──┤ X ├────■─────────────■─────┤ X ├─────■────────────────■───░─┤0     ├
«       ├───┤    │             │     └───┘     │                │   ░ │  QFT │
«q_1: ──┤ X ├────■─────────────■───────────────■────────────────■───░─┤1     ├
«     ┌─┴───┴─┐┌─┴─┐┌───────┐┌─┴─┐

In [5]:
# --- PHASE 3: THE ERASER (Your Custom Decoupling Sequence) ---

# Step 1: Create another visual barrier
qc.barrier()

# Step 2: Define our rotation angle (alpha)
# For this test, we will use pi/4 to rotate the state toward the equator
alpha = np.pi / 4

# Step 3: The Rotation Gate (Ry)
# Apply to the high-frequency target (Qubit 1) to align it
qc.ry(alpha, 1)

# Step 4: The Basis Flip (Hadamard Gate)
# Apply to the target (Qubit 1) to flip it from the X-axis to the Z-axis
qc.h(1)

# Step 5: The Decoupler (CNOT Gate)
# Control = Qubit 0 (Low frequency)
# Target  = Qubit 1 (High frequency)
# This forces the destructive interference to collapse Qubit 1!
qc.cx(0, 1)

# Step 6: Show the updated circuit
print("\n--- Phase 3 Complete ---")
print("Custom Eraser Sequence added successfully!")
print("Here is your updated circuit diagram:")
print(qc.draw('text'))


--- Phase 3 Complete ---
Custom Eraser Sequence added successfully!
Here is your updated circuit diagram:
        ┌───┐   ┌───┐                        ┌───┐                          »
q_0: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├─────────■─────────────■──»
        ├───┤   ├───┤  │                │    ├───┤  ┌───┐  │             │  »
q_1: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├──┤ X ├──■─────────────■──»
     ┌──┴───┴──┐└───┘┌─┴─┐┌──────────┐┌─┴─┐┌─┴───┴─┐└───┘┌─┴─┐┌───────┐┌─┴─┐»
q_2: ┤ Ry(π/2) ├─────┤ X ├┤ Ry(-π/2) ├┤ X ├┤ Ry(0) ├─────┤ X ├┤ Ry(0) ├┤ X ├»
     └─────────┘     └───┘└──────────┘└───┘└───────┘     └───┘└───────┘└───┘»
«       ┌───┐                        ┌───┐                          ░ ┌──────┐»
«q_0: ──┤ X ├────■─────────────■─────┤ X ├─────■────────────────■───░─┤0     ├»
«       ├───┤    │             │     └───┘     │                │   ░ │  QFT │»
«q_1: ──┤ X ├────■─────────────■───────────────■────────────────■───░─┤1     ├»
«     ┌─┴───┴─┐┌─┴─┐┌──────

In [6]:
from qiskit import ClassicalRegister

# --- PHASE 4: THE RECONSTRUCTOR ---

# Step 1: Create a visual barrier
qc.barrier()

# Step 2: The Inverse Eraser
# We run your sequence strictly in reverse.
# The inverse of a CNOT is a CNOT, the inverse of H is H,
# and the inverse of Ry(alpha) is Ry(-alpha)!
qc.cx(0, 1)
qc.h(1)
qc.ry(-alpha, 1)

# Step 3: The Inverse QFT (IQFT)
# We take our original QFT and reverse it to turn frequencies back into spatial pixels.
iqft_circuit = qft_circuit.inverse()
iqft_circuit.name = 'IQFT'
qc.append(iqft_circuit, [0, 1])
qc = qc.decompose('IQFT')

# Step 4: Measurement
# We need to measure the quantum state to see the final image.
# We add 3 classical bits to store the 0s and 1s we read from our 3 qubits.
cr = ClassicalRegister(3, 'c')
qc.add_register(cr)
qc.measure([0, 1, 2], [0, 1, 2])

# Step 5: Show the updated circuit
print("\n--- Phase 4 Complete ---")
print("Reconstructor Sequence and Measurements added successfully!")
print("Here is your fully completed Quantum Circuit:")
print(qc.draw('text'))


--- Phase 4 Complete ---
Reconstructor Sequence and Measurements added successfully!
Here is your fully completed Quantum Circuit:
        ┌───┐   ┌───┐                        ┌───┐                          »
q_0: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├─────────■─────────────■──»
        ├───┤   ├───┤  │                │    ├───┤  ┌───┐  │             │  »
q_1: ───┤ H ├───┤ X ├──■────────────────■────┤ X ├──┤ X ├──■─────────────■──»
     ┌──┴───┴──┐└───┘┌─┴─┐┌──────────┐┌─┴─┐┌─┴───┴─┐└───┘┌─┴─┐┌───────┐┌─┴─┐»
q_2: ┤ Ry(π/2) ├─────┤ X ├┤ Ry(-π/2) ├┤ X ├┤ Ry(0) ├─────┤ X ├┤ Ry(0) ├┤ X ├»
     └─────────┘     └───┘└──────────┘└───┘└───────┘     └───┘└───────┘└───┘»
c: 3/═══════════════════════════════════════════════════════════════════════»
                                                                            »
«       ┌───┐                        ┌───┐                          ░ ┌──────┐»
«q_0: ──┤ X ├────■─────────────■─────┤ X ├─────■────────────────■───░─┤0     ├»
«     

In [8]:
import sys
!{sys.executable} -m pip install qiskit-aer
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# --- PHASE 5: THE LOCAL SIMULATION ---

print("\n--- Phase 5: Running the Simulation ---")

# Step 1: Turn on the virtual quantum computer (simulator)
simulator = AerSimulator()

# Step 2: Compile our circuit so the simulator can understand it
compiled_circuit = transpile(qc, simulator)

# Step 3: Run the circuit 10,000 times to get accurate probabilities
# (Quantum mechanics is probabilistic, so we measure it many times!)
job = simulator.run(compiled_circuit, shots=10000)

# Step 4: Grab the final results
result = job.result()
counts = result.get_counts(qc)

# Step 5: Show the data!
print("Measurement Results (Binary Output):", counts)

# This will pop up a nice bar chart showing our final image data
plot_histogram(counts, title="Reconstructed Quantum Image Data")
plt.show()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.8 MB/s eta 0:00:00

--- Phase 5: Running the Simulation ---
Measurement Results (Binary Output): {'111': 2474, '001': 2495, '010': 2587, '100': 2444}


In [9]:
pip install qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.8/381.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 16.5 MB/s eta 0:00:00


In [12]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
token="<ApiKey-4845f55b-fa83-489e-9fd7-201cf971bc21>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
instance="<CRN>", # Optional
overwrite=True
)

In [21]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
token="<gY4UdgwsLZNi8-h0btzzjMgxgmNig_N_9krR3oEPNFus>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
instance="<crn:v1:bluemix:public:quantum-computing:us-east:a/453360b3786e4b10a065a173d46c6c0f:23257f71-d83a-4438-9def-526d2721398d::>", # Optional
overwrite=True
)

In [23]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# --- PHASE 6: REAL IBM QUANTUM HARDWARE (The Target Fix) ---

print("\n--- Phase 6: Connecting to IBM Quantum ---")

# Step 1: Authenticate
service = QiskitRuntimeService(channel="ibm_quantum_platform", token="gY4UdgwsLZNi8-h0btzzjMgxgmNig_N_9krR3oEPNFus")

# Step 2: Find backend (Modified to use a simulator for faster results)
print("Searching for the least busy quantum simulator...")
backend = service.least_busy(operational=True, simulator=True, min_num_qubits=3)
print(f"Success! We are connecting to: {backend.name}")

# Step 3: THE FIX - Use backend.target instead of backend
print(f"Translating circuit safely for {backend.name}'s raw hardware...")
pm = generate_preset_pass_manager(target=backend.target, optimization_level=1)
isa_circuit = pm.run(qc)

# Step 4: Send the job!
print("Sending job to the quantum processor... (This will be much faster on a simulator!)")
job = backend.run(isa_circuit, shots=1024)

# Step 5: Wait for the result
print(f"Job ID: {job.job_id()}")
print("Waiting for the physical hardware to finish... check your IBM dashboard to see your place in line.")

# Step 6: Grab results
real_result = job.result()
real_counts = real_result.get_counts()

print("\n--- REAL HARDWARE RESULTS ---")
print("Measurement Results (Binary Output):", real_counts)

qiskit_runtime_service._discover_account:WARNING:2026-03-07 17:27:20,333: Loading account with the given token. A saved account will not be used.



--- Phase 6: Connecting to IBM Quantum ---


qiskit_runtime_service.__init__:WARNING:2026-03-07 17:27:22,778: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Searching for the least busy quantum simulator...


qiskit_runtime_service.backends:WARNING:2026-03-07 17:27:23,671: Loading instance: open-instance, plan: open


QiskitBackendNotFoundError: 'No backend matches the criteria.'

In [25]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# --- PHASE 6: REAL IBM QUANTUM HARDWARE (The Primitives Fix) ---

print("\n--- Phase 6: Connecting to IBM Quantum ---")

# Step 1: Authenticate
# Keep your token safe! Paste it here before running.
service = QiskitRuntimeService(channel="ibm_quantum_platform", token="gY4UdgwsLZNi8-h0btzzjMgxgmNig_N_9krR3oEPNFus")

# Step 2: Find backend
print("Searching for the least busy REAL quantum computer...")
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=3)
print(f"Success! We are connecting to: {backend.name}")

# Step 3: Use backend.target to translate
print(f"Translating circuit safely for {backend.name}'s raw hardware...")
pm = generate_preset_pass_manager(target=backend.target, optimization_level=1)
isa_circuit = pm.run(qc)

# Step 4: THE FIX - Send the job using the new Sampler Primitive!
print("Sending job to the quantum processor using SamplerV2... (You will likely wait in a queue!)")
sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=1024)

# Step 5: Wait for the result
print(f"Job ID: {job.job_id()}")
print("Waiting for the physical hardware to finish... check your IBM dashboard to see your place in line.")

# Step 6: Grab results using the new V2 format
real_result = job.result()
# In V2, the data is bundled up. We extract it from our classical register named 'c'
real_counts = real_result[0].data.c.get_counts()

print("\n--- REAL HARDWARE RESULTS ---")
print("Measurement Results (Binary Output):", real_counts)

qiskit_runtime_service._discover_account:WARNING:2026-03-07 17:31:26,561: Loading account with the given token. A saved account will not be used.



--- Phase 6: Connecting to IBM Quantum ---


qiskit_runtime_service.__init__:WARNING:2026-03-07 17:31:29,717: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Searching for the least busy REAL quantum computer...


qiskit_runtime_service.backends:WARNING:2026-03-07 17:31:30,841: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-03-07 17:31:34,051: Using instance: open-instance, plan: open


Success! We are connecting to: ibm_fez
Translating circuit safely for ibm_fez's raw hardware...
Sending job to the quantum processor using SamplerV2... (You will likely wait in a queue!)
Job ID: d6m61u69td6c73amknvg
Waiting for the physical hardware to finish... check your IBM dashboard to see your place in line.

--- REAL HARDWARE RESULTS ---
Measurement Results (Binary Output): {'001': 186, '010': 208, '100': 170, '000': 92, '101': 41, '111': 203, '110': 54, '011': 70}
